In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp04/ex_5.py ──
"""
Shared infrastructure for MLFP04 Exercise 5 — Association Rules.

Contains: synthetic Singapore retail transaction generator, category map,
rule dataclass helpers, and small polars utilities used by every technique
file in ``modules/mlfp04/solutions/ex_5/``.

Technique-specific code (Apriori from scratch, FP-Growth wrapper, rule
evaluation, feature engineering for classification) does NOT belong here —
it lives in the per-technique files.
"""

import asyncio
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import polars as pl

from kailash_ml import ExperimentTracker


setup_environment()

# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp04_ex5_association_rules"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# SINGAPORE RETAIL PRODUCT CATALOGUE
# ════════════════════════════════════════════════════════════════════════
# 25 products grouped to mirror a typical HDB neighbourhood mini-mart basket.

PRODUCTS: list[str] = [
    "bread",
    "butter",
    "milk",
    "eggs",
    "rice",
    "noodles",
    "soy_sauce",
    "cooking_oil",
    "chicken",
    "fish",
    "coffee",
    "tea",
    "sugar",
    "condensed_milk",
    "biscuits",
    "chips",
    "soft_drink",
    "beer",
    "wine",
    "tissue",
    "shampoo",
    "soap",
    "detergent",
    "toothpaste",
    "bananas",
]

CATEGORY_MAP: dict[str, str] = {
    "bread": "breakfast",
    "butter": "breakfast",
    "eggs": "breakfast",
    "milk": "dairy",
    "condensed_milk": "dairy",
    "coffee": "beverage",
    "tea": "beverage",
    "soft_drink": "beverage",
    "sugar": "pantry",
    "rice": "pantry",
    "cooking_oil": "pantry",
    "soy_sauce": "pantry",
    "noodles": "pantry",
    "chicken": "protein",
    "fish": "protein",
    "beer": "alcohol",
    "wine": "alcohol",
    "chips": "snack",
    "biscuits": "snack",
    "bananas": "fruit",
    "shampoo": "personal_care",
    "soap": "personal_care",
    "toothpaste": "personal_care",
    "tissue": "household",
    "detergent": "household",
}

# Co-purchase bundles (items, probability). Models real behaviour:
# kaya-toast breakfast, kopi-C, household replenishment, beer+chips, etc.
BUNDLES: list[tuple[list[str], float]] = [
    (["bread", "butter", "eggs"], 0.15),
    (["coffee", "condensed_milk", "sugar"], 0.12),
    (["rice", "chicken", "soy_sauce"], 0.10),
    (["noodles", "eggs", "soy_sauce"], 0.08),
    (["beer", "chips"], 0.09),
    (["milk", "biscuits"], 0.07),
    (["shampoo", "soap", "toothpaste"], 0.06),
    (["tea", "sugar", "biscuits"], 0.05),
    (["wine", "chips", "biscuits"], 0.04),
    (["cooking_oil", "rice", "fish"], 0.06),
    (["detergent", "tissue", "soap"], 0.05),
    (["bananas", "milk", "eggs"], 0.05),
]

N_TRANSACTIONS_DEFAULT: int = 2500


# ════════════════════════════════════════════════════════════════════════
# TRANSACTION GENERATOR
# ════════════════════════════════════════════════════════════════════════


def _draw_basket(rng: np.random.Generator, bundle_probs: Iterable[float]) -> set[str]:
    """Draw one basket: each bundle fires with its probability, each item in
    a firing bundle is kept with 0.85 probability, plus a Poisson(2) number
    of random impulse items."""
    basket: set[str] = set()
    for (bundle_items, _), prob in zip(BUNDLES, bundle_probs):
        if rng.random() < prob:
            for item in bundle_items:
                if rng.random() < 0.85:
                    basket.add(item)
    n_random = rng.poisson(2)
    if n_random > 0:
        random_items = rng.choice(PRODUCTS, size=int(min(n_random, 5)), replace=False)
        basket.update(random_items)
    return basket


def generate_transactions(
    n: int = N_TRANSACTIONS_DEFAULT,
    seed: int = 42,
) -> list[set[str]]:
    """Generate ``n`` non-empty synthetic Singapore retail transactions.

    Each transaction is a set of product strings. Bundles fire with their
    listed probability; each item inside a firing bundle is kept with 0.85
    probability (random drop-out) so support is noisy. A Poisson number of
    random items is added on top to simulate impulse buys. Empty draws are
    re-drawn, so exactly ``n`` baskets are returned.
    """
    rng = np.random.default_rng(seed)
    base_probs = [prob for _, prob in BUNDLES]
    transactions: list[set[str]] = []
    while len(transactions) < n:
        basket = _draw_basket(rng, base_probs)
        if basket:
            transactions.append(basket)
    return transactions


# Habit model for two-trip shoppers (used by Exercise 5.4). A shopper is a
# "regular" for a bundle with probability HABIT_RATE_MULTIPLIER x the
# bundle's base rate; regulars fire that bundle on HABIT_FIRE_PROB of their
# trips, everyone else on CASUAL_FIRE_PROB of their trips.
HABIT_RATE_MULTIPLIER: float = 1.5
HABIT_FIRE_PROB: float = 0.6
CASUAL_FIRE_PROB: float = 0.03


def generate_shopper_trips(
    n_shoppers: int = N_TRANSACTIONS_DEFAULT,
    seed: int = 42,
) -> tuple[list[set[str]], list[set[str]]]:
    """Two consecutive baskets per shopper: ``(this_trip, next_trip)``.

    Both trips are drawn independently from the SAME per-shopper habit
    profile, so this trip carries real but noisy evidence about the next
    one. The next trip is never used to build features — it only supplies
    a prediction target that is NOT a function of this trip's items.
    """
    rng = np.random.default_rng(seed)
    base = np.array([prob for _, prob in BUNDLES])
    this_trip: list[set[str]] = []
    next_trip: list[set[str]] = []
    while len(this_trip) < n_shoppers:
        habit = rng.random(len(BUNDLES)) < base * HABIT_RATE_MULTIPLIER
        probs = np.where(habit, HABIT_FIRE_PROB, CASUAL_FIRE_PROB)
        first = _draw_basket(rng, probs)
        second = _draw_basket(rng, probs)
        if first:
            this_trip.append(first)
            next_trip.append(second)
    return this_trip, next_trip


def transactions_to_onehot(transactions: list[set[str]]) -> pl.DataFrame:
    """One-row-per-transaction boolean matrix (columns = sorted PRODUCTS).

    Polars-native. mlxtend needs pandas, so callers convert with
    ``.to_pandas()`` at the mlxtend call site only.
    """
    all_items = sorted(PRODUCTS)
    rows = [{item: (item in txn) for item in all_items} for txn in transactions]
    return pl.DataFrame(rows)


def product_frequency(transactions: Iterable[set[str]]) -> dict[str, int]:
    """Count how many transactions contain each product."""
    counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            counts[item] += 1
    return dict(counts)


def print_transaction_summary(transactions: list[set[str]]) -> None:
    """One-liner summary + top 10 product frequency. Used by every file."""
    avg_basket = float(np.mean([len(t) for t in transactions]))
    print("=== Synthetic Singapore Retail Transactions ===")
    print(f"  Transactions: {len(transactions):,}")
    print(f"  Products:     {len(PRODUCTS)}")
    print(f"  Avg basket:   {avg_basket:.1f} items")

    freq = product_frequency(transactions)
    n = len(transactions)
    print("\n  Top 10 products by frequency:")
    for item, count in sorted(freq.items(), key=lambda kv: -kv[1])[:10]:
        print(f"    {item:<20} {count:>5} ({count / n:.1%})")


# ════════════════════════════════════════════════════════════════════════
# RULE HELPERS
# ════════════════════════════════════════════════════════════════════════


def format_itemset(items: Iterable[str]) -> str:
    """Deterministic pretty-print of a frozenset of items."""
    return ", ".join(sorted(items))


def categorise_rule(
    antecedent: frozenset[str],
    consequent: frozenset[str],
) -> tuple[set[str], set[str], str]:
    """Return (antecedent_categories, consequent_categories, relation_type)."""
    ant_cats = {CATEGORY_MAP.get(item, "other") for item in antecedent}
    con_cats = {CATEGORY_MAP.get(item, "other") for item in consequent}
    if ant_cats == con_cats:
        rel = "within-category complement"
    elif ant_cats & con_cats:
        rel = "cross-category with overlap"
    else:
        rel = "cross-category association"
    return ant_cats, con_cats, rel


def rules_to_polars(rules: list[dict]) -> pl.DataFrame:
    """Convert a list of rule dicts into a polars DataFrame for plotting."""
    return pl.DataFrame(
        {
            "antecedent": [format_itemset(r["antecedent"]) for r in rules],
            "consequent": [format_itemset(r["consequent"]) for r in rules],
            "support": [float(r["support"]) for r in rules],
            "confidence": [float(r["confidence"]) for r in rules],
            "lift": [float(r["lift"]) for r in rules],
        }
    )


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ML EXPERIMENT TRACKER — shared by every association-rules technique
# ════════════════════════════════════════════════════════════════════════
# Every M4 ex_5 lesson logs its mining + rule-evaluation + classifier runs
# into a single SQLite store so students can compare techniques side by side
# (Apriori vs FP-Growth runtime, rule-quality at varying thresholds,
# baseline vs rule-enhanced classifier metrics) after the lesson group ends.
# Mirrors m4_clustering_zoo / m4_dimreduction_zoo / m4_anomaly_zoo from
# ex_1 / ex_3 / ex_4.
#
# IMPORTANT: this store is SEPARATE from the other M4 stores. Per the
# SQLite contention trap (.session-notes), running two ExperimentTracker
# writers against the same SQLite file concurrently triggers `disk I/O
# error`. Each exercise group gets its own DB.

ASSOC_DB = "sqlite:///mlfp04_ex5_assoc_rules.db"
ASSOC_EXPERIMENT_NAME = "m4_assoc_rules_zoo"

_TRACKER_KEY_RE = re.compile(r"[^a-zA-Z0-9_.\-]")


def _slug(text: str) -> str:
    """Coerce free-form prose into a tracker-key-safe slug.

    ExperimentTracker enforces ``^[a-zA-Z_][a-zA-Z0-9_.\\-]*$`` on metric
    names. Replace forbidden characters with underscores; ensure a leading
    alpha character; collapse repeats.
    """
    s = _TRACKER_KEY_RE.sub("_", text.strip())
    s = re.sub(r"_+", "_", s).strip("_") or "metric"
    if not s[0].isalpha() and s[0] != "_":
        s = f"m_{s}"
    return s


def _finite(x: float) -> float:
    """Tracker rejects NaN/inf via MetricValueError; coerce to 0.0.

    Association-rule metrics (lift, conviction) can collapse to inf when
    confidence saturates at 1.0; aggregate scalars over empty filter sets
    can NaN. Every emit through this guard.
    """
    return float(x) if x == x and x not in (float("inf"), float("-inf")) else 0.0


async def _setup_engines_async() -> tuple[ExperimentTracker, str]:
    """Open the association-rules ExperimentTracker."""
    tracker = await ExperimentTracker.create(store_url=ASSOC_DB)
    return tracker, ASSOC_EXPERIMENT_NAME


def setup_engines() -> tuple[ExperimentTracker, str]:
    """Sync wrapper. Returns (tracker, experiment_name)."""
    return asyncio.run(_setup_engines_async())


def teardown_engines(tracker: ExperimentTracker) -> None:
    """Drain the aiosqlite worker threads before the script returns.

    kailash's AsyncSQLitePool spawns NON-DAEMON aiosqlite worker threads on
    first pool use. Python 3.13's ``Py_FinalizeEx`` joins non-daemon threads
    BEFORE running ``atexit`` handlers, so an atexit-based close runs too
    late — the interpreter hangs forever in ``wait_for_thread_shutdown``
    waiting on workers stuck in ``queue.get()``.

    Solutions MUST call ``teardown_engines(tracker)`` after the REFLECTION
    block (or use it via ``try: … finally: teardown_engines(tracker)``).
    See ``rules/patterns.md`` § "Async Resource Cleanup": real cleanup is
    the caller's responsibility; ``close()`` must NOT live in ``__del__``.
    """
    asyncio.run(tracker.close())


async def _track_run_async(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Log one lesson's run: scalar metrics + optional per-step series."""
    async with tracker.track(experiment=exp_name, run_name=run_name) as run:
        await run.log_params({k: str(v) for k, v in params.items()})
        for name, value in scalar_metrics.items():
            await run.log_metric(_slug(name), _finite(value))
        if series_metrics:
            for name, values in series_metrics.items():
                slug = _slug(name)
                for step, value in enumerate(values, start=1):
                    await run.log_metric(slug, _finite(value), step=step)


def track_run(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Sync wrapper for logging a single technique's run."""
    asyncio.run(
        _track_run_async(
            tracker, exp_name, run_name, params, scalar_metrics, series_metrics
        )
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 5.1: Apriori Algorithm from Scratch
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Implement the Apriori algorithm for frequent itemset mining
#   - Apply the anti-monotone pruning principle by hand
#   - Generate candidate k-itemsets from frequent (k-1)-itemsets
#   - Count support with a single pass per level
#   - Understand why Apriori scales beyond brute-force enumeration
#
# PREREQUISITES:
#   - MLFP04 Exercise 1 (clustering — pattern discovery without labels)
#   - Basic set theory (subset, superset, intersection)
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — why Apriori works (anti-monotone pruning)
#   2. Build — implement `apriori()` and `_generate_candidates()`
#   3. Train — run it on 2,500 Singapore retail transactions
#   4. Visualise — L1 -> L2 -> L3 ladder and top frequent itemsets
#   5. Apply — supermarket shelf layout optimisation
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from collections import defaultdict

import polars as pl


# ── Kailash-ML ExperimentTracker — every association-rules run logs here ─
tracker, exp_name = setup_engines()



## THEORY — Why Apriori Works


In [ ]:
# With 25 products there are 2^25 ~= 33 million possible itemsets. A naive
# algorithm would test every one of them against every transaction. Apriori
# avoids that with a single observation:
#
#   ANTI-MONOTONE PRINCIPLE
#   If an itemset X is INFREQUENT, then every superset of X is ALSO
#   infrequent. Reason: if fewer than min_count baskets contain X, then
#   even fewer baskets contain X plus any extra item.
#
# Consequence: once L_k (frequent k-itemsets) is known, we only need to
# form candidates for L_{k+1} by joining L_k with itself AND requiring
# every (k)-subset of the new candidate to already be in L_k. Any itemset
# with an infrequent subset is pruned before it is ever counted.
#
# COST:
#   Level 1: scan once, count single items.  O(N * |I|)
#   Level k: generate C_k from L_{k-1}, scan to count, filter.
#   Terminates when L_k is empty.
#
# On a 2,500-transaction basket with 25 products, Apriori evaluates on
# the order of ~1,000 candidates — a 30,000x reduction from brute force.



## TASK 2 — BUILD: Apriori from scratch


Applies the anti-monotone pruning rule: every (k-1)-subset of a
    candidate MUST already be in `prev_level`, otherwise drop the
    candidate without ever counting it.


Returns ``{itemset: support}`` for every itemset whose support meets
    ``min_support``. Support is the fraction of baskets containing the
    itemset.


In [ ]:
def _generate_candidates(
    prev_level: list[frozenset[str]],
    k: int,
) -> list[frozenset[str]]:
    prev_set = set(prev_level)
    candidates: set[frozenset[str]] = set()

    for i, a in enumerate(prev_level):
        for b in prev_level[i + 1 :]:
            # TODO: union the two itemsets. Hint: use the | operator.
            union = ____
            if len(union) != k:
                continue
            # TODO: anti-monotone check — every (k-1)-subset of `union`
            # must already be frequent (i.e. in prev_set). Build each
            # subset by removing one item (union - frozenset([item])) and
            # test membership with `in`; combine the tests with all(...).
            all_subsets_frequent = ____
            if all_subsets_frequent:
                candidates.add(union)

    return list(candidates)


def apriori(
    transactions: list[set[str]],
    min_support: float,
    verbose: bool = True,
) -> dict[frozenset[str], float]:
    n = len(transactions)
    min_count = min_support * n

    # ── L1: frequent single items ────────────────────────────────────
    item_counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            item_counts[item] += 1

    freq_itemsets: dict[frozenset[str], float] = {}
    current_level: list[frozenset[str]] = []
    for item, count in item_counts.items():
        if count >= min_count:
            fs = frozenset([item])
            # TODO: record the support (count / n) in freq_itemsets and
            # append `fs` to current_level so the next level can join it.
            ____
            ____

    if verbose:
        print(f"  L1: {len(current_level)} frequent items (min_support={min_support})")

    # ── L2, L3, ... Lk: joined + pruned candidates ───────────────────
    k = 2
    while current_level:
        # TODO: generate candidates for level k from the current level.
        # Hint: the helper you completed above.
        candidates = ____
        if not candidates:
            break

        candidate_counts: dict[frozenset[str], int] = defaultdict(int)
        for txn in transactions:
            txn_frozen = frozenset(txn)
            for candidate in candidates:
                if candidate.issubset(txn_frozen):
                    candidate_counts[candidate] += 1

        current_level = []
        for candidate, count in candidate_counts.items():
            if count >= min_count:
                # TODO: keep this candidate — store its support and seed
                # current_level for the next iteration (same as L1).
                ____
                ____

        if verbose:
            print(f"  L{k}: {len(current_level)} frequent {k}-itemsets")
        k += 1

    return freq_itemsets



## TASK 3 — TRAIN: run Apriori on Singapore retail baskets


In [ ]:
transactions = generate_transactions(n=2500, seed=42)
print_transaction_summary(transactions)

print("\n=== Apriori (from scratch) ===")
MIN_SUPPORT = 0.03
frequent_itemsets = apriori(transactions, min_support=MIN_SUPPORT)
print(f"\n  Total frequent itemsets: {len(frequent_itemsets)}")



### Checkpoint


In [ ]:
assert len(frequent_itemsets) > 0, "Apriori should find at least one frequent itemset"
n_txn = len(transactions)
for itemset, support in list(frequent_itemsets.items())[:5]:
    actual_count = sum(1 for t in transactions if itemset.issubset(frozenset(t)))
    actual_support = actual_count / n_txn
    assert (
        abs(actual_support - support) < 0.005
    ), f"Computed support {support:.4f} disagrees with actual {actual_support:.4f}"
    assert (
        support >= MIN_SUPPORT - 0.001
    ), f"Itemset with support {support:.4f} should meet min_support={MIN_SUPPORT}"
print("\n[ok] Checkpoint passed — Apriori computes correct support values\n")



## TASK 4 — VISUALISE the L1 -> Lk ladder and top itemsets


In [ ]:
sorted_itemsets = sorted(frequent_itemsets.items(), key=lambda kv: -kv[1])

print("Top 15 frequent itemsets by support:")
print(f"  {'Itemset':<45} {'Support':>8}")
print("  " + "-" * 55)
for itemset, support in sorted_itemsets[:15]:
    print(f"  {format_itemset(itemset):<45} {support:>8.4f}")

# Export to polars for any downstream visualisation
top_df = pl.DataFrame(
    {
        "itemset": [format_itemset(s) for s, _ in sorted_itemsets[:30]],
        "size": [len(s) for s, _ in sorted_itemsets[:30]],
        "support": [float(v) for _, v in sorted_itemsets[:30]],
    }
)
top_df.write_csv(OUTPUT_DIR / "apriori_top_itemsets.csv")
print(f"\n  Saved: {OUTPUT_DIR / 'apriori_top_itemsets.csv'}")



### Visualisation


In [ ]:
import plotly.graph_objects as go  # noqa: E402
from plotly.subplots import make_subplots  # noqa: E402

# (A) The Apriori ladder: how many itemsets survive at each level k, and
# (B) the top 15 itemsets coloured by size so the L2/L3 co-purchase
# structure stands out from the single-item staples.
level_counts: defaultdict[int, int] = defaultdict(int)
for itemset in frequent_itemsets:
    level_counts[len(itemset)] += 1
levels = sorted(level_counts)
size_colours = {1: "#636EFA", 2: "#EF553B", 3: "#00CC96", 4: "#AB63FA"}

fig_apriori = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.35, 0.65],
    subplot_titles=[
        "Frequent itemsets per level (L1 -> Lk)",
        "Top 15 itemsets by support (colour = itemset size)",
    ],
)
fig_apriori.add_trace(
    go.Bar(
        x=[f"L{k}" for k in levels],
        y=[level_counts[k] for k in levels],
        marker_color=[size_colours.get(k, "#7F7F7F") for k in levels],
        text=[level_counts[k] for k in levels],
        textposition="outside",
        showlegend=False,
    ),
    row=1,
    col=1,
)
top15 = sorted_itemsets[:15]
fig_apriori.add_trace(
    go.Bar(
        x=[float(v) for _, v in top15],
        y=[format_itemset(s) for s, _ in top15],
        orientation="h",
        marker_color=[size_colours.get(len(s), "#7F7F7F") for s, _ in top15],
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig_apriori.update_yaxes(autorange="reversed", row=1, col=2)
fig_apriori.update_layout(
    title=f"Apriori at min_support={MIN_SUPPORT}: the pruned ladder",
    height=500,
    width=1100,
)
apriori_path = OUTPUT_DIR / "01_apriori_ladder.html"
fig_apriori.write_html(str(apriori_path))
print(f"[viz] Apriori ladder + top itemsets: {apriori_path}")

# INTERPRETATION: The L1 level is dense (most of the 25 products appear in
# >= 3% of baskets) because Singapore mini-marts stock fast-moving staples.
# The interesting content is at L2 and L3 — that's where co-purchase
# structure (coffee + condensed milk + sugar) appears. If Apriori stops
# early at L2 for your dataset, your min_support is probably too high.



## TASK 5 — APPLY: supermarket shelf layout optimisation


In [ ]:
# SCENARIO: A Singapore supermarket chain operates ~200 neighbourhood
# outlets, each stocking roughly 8,000 SKUs in 1,200 sqft of HDB floor
# space. Shelf real estate is the single largest cost driver after payroll.
#
# A merchandising analyst wants to find frequent 2- and 3-itemsets so that
# physically adjacent shelving can be re-planned: items that appear together
# in >= 3% of baskets are candidates for cross-category co-location (e.g.,
# moving condensed milk next to the kopi aisle, rather than in the dairy
# fridge on the far wall).
#
# WHY APRIORI FITS:
#   - The item universe (~8K SKUs) is too large for brute-force enumeration
#   - Store managers need the frequent-itemset list monthly, not live
#   - The anti-monotone pruning removes ~99.9% of candidate itemsets
#   - Output is directly interpretable — each row is a physical product set
#
# BUSINESS IMPACT (illustrative assumptions, not measured figures): if
# co-locating the top 50 cross-category frequent pairs lifted basket
# size by 2%, a chain with S$250M annual GMV would gain ~S$5M of sales
# for zero marginal inventory cost. Validate the lift with a store-level
# A/B test before re-planning every outlet. The Apriori run takes
# seconds; the merchandising re-plan takes a weekend.
#
# LIMITATIONS:
#   - Apriori re-scans the transaction log at every level; for 100K+ txns
#     with thousands of SKUs, FP-Growth (Exercise 5.2) is much faster
#   - Support alone is not actionable — you also need confidence + lift
#     (Exercise 5.3) before deciding which pairings are worth the shelf move



## TRACK — Log this lesson's run to the kailash-ml ExperimentTracker


In [ ]:
# Every M4 ex_5 lesson logs into the SAME experiment ('m4_assoc_rules_zoo')
# so Apriori / FP-Growth / rule-evaluation / rule-features can be compared
# from one SQLite store after the lesson group ends. Series = per-level
# itemset counts (the L1->Lk ladder); scalars = headline counts + supports.

# Per-level size series — the visible "ladder" Apriori climbs.
size_to_count: defaultdict[int, int] = defaultdict(int)
for itemset in frequent_itemsets:
    size_to_count[len(itemset)] += 1
max_k = max(size_to_count) if size_to_count else 0
ladder = [size_to_count[k] for k in range(1, max_k + 1)]
support_values = [float(s) for s in frequent_itemsets.values()]

track_run(
    tracker,
    exp_name,
    run_name=____,  # TODO: a stable name, e.g. "apriori_from_scratch"
    params={
        "algorithm": "apriori",
        "implementation": "from_scratch",
        "n_transactions": len(transactions),
        "n_products": len(PRODUCTS),
        "min_support": MIN_SUPPORT,
        "max_k_reached": max_k,
    },
    scalar_metrics={
        "n_frequent_itemsets": float(len(frequent_itemsets)),
        # TODO: how many singletons (size 1) and pairs (size 2) were found.
        # Hint: size_to_count maps itemset size -> count; use .get(k, 0).
        "n_singletons_L1": ____,
        "n_pairs_L2": ____,
        "n_triples_L3": float(size_to_count.get(3, 0)),
        "max_k_reached": float(max_k),
        "max_support": float(max(support_values) if support_values else 0.0),
        "min_support_observed": float(min(support_values) if support_values else 0.0),
        "mean_support": float(
            sum(support_values) / len(support_values) if support_values else 0.0
        ),
    },
    series_metrics={"frequent_count_by_level": [float(c) for c in ladder]},
)
print(f"  [tracked] Apriori ladder + headline counts logged to {exp_name}\n")



## DESTINATION-FIRST CLOSE — the mlxtend.frequent_patterns.apriori one-liner


In [ ]:
# This lesson hand-rolled the level-wise scan, the candidate joiner, and
# the anti-monotone pruner — ~75 lines of structure to internalise WHY
# Apriori scales beyond brute force. The production destination is one
# function call. ``mlxtend.frequent_patterns.apriori`` runs the same
# algorithm on a one-hot pandas DataFrame and returns a pandas table
# (mlxtend needs pandas, so we convert only at this call site).
# Lesson 02 (FP-Growth) then swaps the algorithm without changing the
# call shape — both speak the same itemset-table contract.

from mlxtend.frequent_patterns import apriori as mlx_apriori  # noqa: E402


onehot_pd = transactions_to_onehot(transactions).to_pandas().astype(bool)
# TODO: call mlxtend's apriori with the SAME support threshold you used
# above so the two results are comparable.
mlx_frequent = mlx_apriori(onehot_pd, min_support=____, use_colnames=True)
print(
    f"  mlxtend.apriori(min_support={MIN_SUPPORT}): "
    f"{len(mlx_frequent)} frequent itemsets in 1 call"
)
hand_set = {frozenset(items) for items in frequent_itemsets}
mlx_set = {frozenset(items) for items in mlx_frequent["itemsets"].tolist()}
print(f"  Hand-rolled: {len(hand_set)}  mlxtend: {len(mlx_set)}")
print(
    f"  Intersection: {len(hand_set & mlx_set)}  symmetric diff: {len(hand_set ^ mlx_set)}"
)
print()
print("  Same algorithm, same result — the hand-rolled walk-through was")
print("  the WHY; mlxtend.apriori is the destination you ship to prod.\n")



## REFLECTION


[x] Implemented the Apriori algorithm from scratch
  [x] Applied the anti-monotone pruning principle in _generate_candidates()
  [x] Counted support with one pass per level against 2,500 baskets
  [x] Visualised the pruned L1 -> Lk ladder and the top itemsets
  [x] Identified a production scenario (SG grocery shelf layout) where
      Apriori is the economically optimal choice
  [x] Compared the hand-rolled implementation against mlxtend.apriori —
      one call, same itemsets, ready for production

  KEY INSIGHT: Pruning > optimisation. The reason Apriori scales is not
  clever data structures — it is the mathematical observation that
  infrequent sets cannot become frequent when you add items. Every
  frequent-itemset miner you'll meet later (FP-Growth, ECLAT) uses the
  same principle, just with different data layouts.

  Next: 02_fp_growth.py — use mlxtend's FP-Growth (no candidate
  generation) and compare it against the Apriori output from this file.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

# Drain the aiosqlite worker threads so Py_Finalize doesn't hang.
teardown_engines(tracker)

